# Debugging Pandas

### Loading Libraries

In [ ]:
# Numerical Computing
import numpy as np

# Data Manipualtion
import pandas as pd

# Data Visualisation
import seaborn as sns
import matplotlib.pyplot as plt

# PyArrow
import pyarrow as pa

# URL
import urllib.request

In [ ]:
with open('/Users/joaquinromero/Desktop/EP2/data/devilclean.txt', 'r') as f:
    for i in range(10):
        print(repr(f.readline()))

In [ ]:
url = (
    'https://github.com/mattharrison/datasets/raw/master'
    '/data/dirtydevil.txt'
)

local_filename = '/Users/joaquinromero/Desktop/EP2/data/devilclean.txt'

urllib.request.urlretrieve(url, local_filename)

In [ ]:
with open(local_filename, 'r') as file:
    lines = file.readlines()

with open(local_filename, 'w') as file:
    for i, line in enumerate(lines):

        # elimina metadata del USGS
        if i < 34 or i == 35:
            continue

        file.write(line)

In [ ]:
with open(local_filename, 'r') as f:
    for i in range(5):
        print(repr(f.readline()))

In [ ]:
def to_denver_time(df_, time_col, tz_col):

    return (

        df_

        .assign(**{tz_col: df_[tz_col].replace('MDT', 'MST7MDT')})

        .groupby(tz_col)[time_col]

        .transform(

            lambda s:

                pd.to_datetime(s)

                .dt.tz_localize(s.name, ambiguous=True)

                .dt.tz_convert('America/Denver')

        )

    )

def tweak_river(df_):

    return (

        df_

        .assign(

            datetime=to_denver_time(df_, 'datetime', 'tz_cd')

        )

        .rename(

            columns={

                '144166_00060': 'cfs',

                '144167_00065': 'gage_height'

            }

        )

        .loc[

            :,

            [

                'datetime',

                'agency_cd',

                'site_no',

                'tz_cd',

                'cfs',

                'gage_height'

            ]

        ]

    )

In [ ]:
dd = tweak_river(df)

print(dd)

In [ ]:
dd2 = pd.read_json(
    dd.to_json(),
    dtype_backend='pyarrow'
)

dd.equals(dd2)

In [ ]:
print(dd.eq(dd2))